<a href="https://colab.research.google.com/github/dtcwee/AdelaideHousePrices/blob/main/AdelaideHousePrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adelaide House Prices
Collect, Collate, and Display house price time series data for Adelaide, South Australia.

# Phase 1: In-Memory Data Acquisition
Fetch Suburb Geometries and list of links to house price data.

In [ ]:
import requests
import zipfile
import io
import pandas as pd
import geopandas as gpd

# 1. Fetch Suburb Geometries (GeoJSON)
zip_url = "https://www.dptiapps.com.au/dataportal/Suburbs_geojson.zip"
r = requests.get(zip_url)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    # Load the specified GeoJSON file
    geojson_name = 'Suburbs_GDA2020.geojson'
    with z.open(geojson_name) as f:
        adelaide_gdf = gpd.read_file(f)

# Debug trace for GeoJSON
print(f"Loaded GeoJSON: {len(adelaide_gdf)} features")
display(adelaide_gdf.head())

# 2. Fetch Time Series Metadata (JSON API)
api_url = "https://data.sa.gov.au/data/api/3/action/package_show?id=0d447195-1158-4a3c-8cc7-0e333b87eb72"
package_data = requests.get(api_url).json()
resources = package_data['result']['resources']

# Filter for the .xlsx files
excel_links = [res['url'] for res in resources if res['format'].lower() == 'xlsx']

# Debug trace for Excel links
print(f"Found {len(excel_links)} Excel links")
display(excel_links[:5]) # Display first 5 links for brevity

Loaded GeoJSON: 1895 features


,postcode,suburb,suburb_number,legalstartdate,shape_Length,shape_Area,geometry
0,0872,AMATA,87206,NaT,0.258469,0.003274,"MULTIPOLYGON (((131.21648 -26.11913, 131.14124..."
1,0872,ANANGU PITJANTJATJARA YANKUNYTJATJARA,87205,NaT,16.039186,9.288729,"MULTIPOLYGON (((132.9773 -25.99855, 133.05764 ..."
2,0872,AYERS RANGE SOUTH,87202,2013-04-26,1.466852,0.117333,"MULTIPOLYGON (((133.45801 -25.99855, 133.4583 ..."
3,0872,DE ROSE HILL,87201,2013-04-26,1.685673,0.167839,"MULTIPOLYGON (((133.42464 -26.40869, 133.42602..."
4,0872,IWANTJA,87209,NaT,0.143976,0.001144,"MULTIPOLYGON (((133.30664 -26.99308, 133.30003..."


Found 40 Excel links


['https://data.sa.gov.au/data/dataset/0d447195-1158-4a3c-8cc7-0e333b87eb72/resource/3a05b4ad-2dfb-4ba6-984c-48676ea0add1/download/lsg_stats_2026_q1.xlsx',
 'https://data.sa.gov.au/data/dataset/0d447195-1158-4a3c-8cc7-0e333b87eb72/resource/a619b176-454f-48e1-aa14-8b9f551491ae/download/lsg_stats_2025_q4.xlsx',
 'https://data.sa.gov.au/data/dataset/0d447195-1158-4a3c-8cc7-0e333b87eb72/resource/cc293e5c-6dac-4c39-b5bb-ba24a5f9ff81/download/lsg_stats_2025_q3.xlsx',
 'https://data.sa.gov.au/data/dataset/0d447195-1158-4a3c-8cc7-0e333b87eb72/resource/79d8de58-1b68-4125-a1cd-d116c49ae95f/download/lsg_stats_2025_q2.xlsx',
 'https://data.sa.gov.au/data/dataset/0d447195-1158-4a3c-8cc7-0e333b87eb72/resource/d83bd5ec-e9f0-4e4b-b894-ff95217fb7de/download/lsg_stats_2025_q1.xlsx']

# Phase 2: Consolidate "Human-Readable" Excel Data
## 2.1 Clean median data

In [ ]:
def clean_median_data(url):
    # Read excel, using header=0 (0-indexed, meaning the 1st row)
    df = pd.read_excel(url, header=0)

    # Melt the DataFrame to transform from wide to long format
    # Identify columns that represent 'Sales' and 'Median' data for different quarters
    id_vars = ['City', 'Suburb']
    value_vars_sales = [col for col in df.columns if 'Sales' in col]
    value_vars_median = [col for col in df.columns if 'Median' in col and 'Median Change' not in col]

    # Melt sales data
    df_sales = df[id_vars + value_vars_sales].melt(id_vars=id_vars, var_name='QuarterYear_Sales', value_name='Sales')
    df_sales['QuarterYear'] = df_sales['QuarterYear_Sales'].str.replace('Sales ', '')

    # Melt median data
    df_median = df[id_vars + value_vars_median].melt(id_vars=id_vars, var_name='QuarterYear_Median', value_name='Median Price')
    df_median['QuarterYear'] = df_median['QuarterYear_Median'].str.replace('Median ', '')

    # Merge sales and median data back together on Suburb and QuarterYear
    # First, handle the potential 'Median Change' column if it exists and needs to be kept.
    # For simplicity, let's just focus on Sales and Median Price and join them.
    # We need to make sure 'QuarterYear' is consistent across both melts for merging.
    merged_df = pd.merge(df_sales, df_median, on=['City', 'Suburb', 'QuarterYear'], how='outer')

    # Extract Quarter and Year using regex
    # Pattern: (digit Q) (year)
    # The regex r'(\d+Q)\s+(\d{4})' captures '1Q' and '2025'
    # Use expand=True to create new columns
    quarter_year_extracted = merged_df['QuarterYear'].str.extract(r'(\d+Q)\s+(\d{4})', expand=True)
    merged_df['Quarter'] = quarter_year_extracted[0]
    merged_df['Year'] = pd.to_numeric(quarter_year_extracted[1], errors='coerce') # Coerce errors to NaN for non-numeric years

    # Drop rows where Quarter or Year could not be determined
    merged_df = merged_df.dropna(subset=['Quarter', 'Year'])
    merged_df['Year'] = merged_df['Year'].astype(int) # Convert Year to int after dropping NaNs

    # Convert Quarter and Year to a specific date (e.g., end of quarter for consistency)
    def get_quarter_end_date(row):
        quarter_map = {'1Q': 3, '2Q': 6, '3Q': 9, '4Q': 12}
        month = quarter_map.get(row['Quarter'])
        if month is None:
            return pd.NaT
        return pd.to_datetime(f"{row['Year']}-{month:02d}-01") + pd.offsets.QuarterEnd(0)

    merged_df['Date'] = merged_df.apply(get_quarter_end_date, axis=1)
    merged_df = merged_df.dropna(subset=['Date']) # Drop rows where date could not be created

    # Convert 'Median Price' to numeric, coercing errors to NaN
    merged_df['Median Price'] = pd.to_numeric(merged_df['Median Price'], errors='coerce')

    # Select and reorder relevant columns
    # Filter for Metropolitan Adelaide only if the sheet contains regional data (this step will be later)
    return merged_df[['City', 'Suburb', 'Date', 'Sales', 'Median Price']]

# Inspect columns of a sample DataFrame to diagnose the KeyError
print("Columns of the first Excel file after adjusting header:")
sample_df = clean_median_data(excel_links[0])
print(sample_df.columns)
display(sample_df.head())

# Consolidate all years into one long-form DataFrame
all_data = pd.concat([clean_median_data(link) for link in excel_links])
all_data = all_data.drop_duplicates(subset=['Suburb', 'Date'])

Columns of the first Excel file after adjusting header:
Index(['City', 'Suburb', 'Date', 'Sales', 'Median Price'], dtype='object')


,City,Suburb,Date,Sales,Median Price
0,ADELAIDE,ADELAIDE,2025-03-31,6.0,1450000.0
1,ADELAIDE,ADELAIDE,2026-03-31,8.0,1585000.0
2,ADELAIDE,NORTH ADELAIDE,2025-03-31,9.0,2282500.0
3,ADELAIDE,NORTH ADELAIDE,2026-03-31,10.0,2480000.0
4,ADELAIDE HILLS,ALDGATE,2025-03-31,5.0,1600000.0


## 2.2 Merge data

In [ ]:
import pandas as pd
import geopandas as gpd

# Save the GeoDataFrame with unique geometries
adelaide_gdf.to_file('adelaide_suburbs.geojson', driver='GeoJSON')
print("Unique suburb geometries exported to 'adelaide_suburbs.geojson' successfully.")

# Prepare suburb_info for merging suburb_number
# Select unique suburb and suburb_number from adelaide_gdf
suburb_info = adelaide_gdf[['suburb', 'suburb_number']].drop_duplicates(subset=['suburb'])

# Merge all_data with suburb_info to add suburb_number
# Note: 'Suburb' in all_data, 'suburb' in suburb_info
all_data_with_suburb_number = pd.merge(all_data, suburb_info, left_on='Suburb', right_on='suburb', how='left')

# Drop the redundant 'suburb' column from the merge
all_data_with_suburb_number = all_data_with_suburb_number.drop(columns=['suburb'])

# Save the tabular property data (all_data_with_suburb_number) to a CSV
all_data_with_suburb_number.to_csv('adelaide_property_data.csv', index=False)
print("Tabular property data exported to 'adelaide_property_data.csv' successfully.")

# --- Verification ---
# Load and display unique suburb geometries
loaded_suburbs_gdf = gpd.read_file('adelaide_suburbs.geojson')
print(f"\nLoaded unique suburb GeoDataFrame: {len(loaded_suburbs_gdf)} features")
print("Loaded GeoDataFrame Info:")
loaded_suburbs_gdf.info()
print("\nFirst 5 rows of loaded_suburbs_gdf:")
display(loaded_suburbs_gdf.head())

# Load and display tabular property data
loaded_property_df = pd.read_csv('adelaide_property_data.csv', parse_dates=['Date'])
print(f"\nLoaded property DataFrame: {len(loaded_property_df)} rows")
print("Loaded DataFrame Info:")
loaded_property_df.info()
print("\nFirst 5 rows of loaded_property_df:")
display(loaded_property_df.head())

Unique suburb geometries exported to 'adelaide_suburbs.geojson' successfully.
Tabular property data exported to 'adelaide_property_data.csv' successfully.

Loaded unique suburb GeoDataFrame: 1895 features
Loaded GeoDataFrame Info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1895 entries, 0 to 1894
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   postcode        1895 non-null   object        
 1   suburb          1895 non-null   object        
 2   suburb_number   1895 non-null   int32         
 3   legalstartdate  1878 non-null   datetime64[ms]
 4   shape_Length    1895 non-null   float64       
 5   shape_Area      1895 non-null   float64       
 6   geometry        1895 non-null   geometry      
dtypes: datetime64[ms](1), float64(2), geometry(1), int32(1), object(2)
memory usage: 96.4+ KB

First 5 rows of loaded_suburbs_gdf:


,postcode,suburb,suburb_number,legalstartdate,shape_Length,shape_Area,geometry
0,0872,AMATA,87206,NaT,0.258469,0.003274,"MULTIPOLYGON (((131.21648 -26.11913, 131.14124..."
1,0872,ANANGU PITJANTJATJARA YANKUNYTJATJARA,87205,NaT,16.039186,9.288729,"MULTIPOLYGON (((132.9773 -25.99855, 133.05764 ..."
2,0872,AYERS RANGE SOUTH,87202,2013-04-26,1.466852,0.117333,"MULTIPOLYGON (((133.45801 -25.99855, 133.4583 ..."
3,0872,DE ROSE HILL,87201,2013-04-26,1.685673,0.167839,"MULTIPOLYGON (((133.42464 -26.40869, 133.42602..."
4,0872,IWANTJA,87209,NaT,0.143976,0.001144,"MULTIPOLYGON (((133.30664 -26.99308, 133.30003..."



Loaded property DataFrame: 21284 rows
Loaded DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21284 entries, 0 to 21283
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   City           21284 non-null  object        
 1   Suburb         21284 non-null  object        
 2   Date           21284 non-null  datetime64[ns]
 3   Sales          6626 non-null   float64       
 4   Median Price   17716 non-null  float64       
 5   suburb_number  20340 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 997.8+ KB

First 5 rows of loaded_property_df:


,City,Suburb,Date,Sales,Median Price,suburb_number
0,ADELAIDE,ADELAIDE,2025-03-31,6.0,1450000.0,500001.0
1,ADELAIDE,ADELAIDE,2026-03-31,8.0,1585000.0,500001.0
2,ADELAIDE,NORTH ADELAIDE,2025-03-31,9.0,2282500.0,500601.0
3,ADELAIDE,NORTH ADELAIDE,2026-03-31,10.0,2480000.0,500601.0
4,ADELAIDE HILLS,ALDGATE,2025-03-31,5.0,1600000.0,515401.0


## 2.3  (Optional) Download Data Files

To download the 2 files you have created find `adelaide_suburbs.geojson` and `adelaide_property_data.csv` in the file browser on the left-hand side of your Colab interface (the folder icon). You can right-click on the file name and select 'Download'.

In [ ]:
from google.colab import files

# Specify the filename to download
filename = 'adelaide_property_data.geojson'

try:
    files.download(filename)
    print(f"'{filename}' downloaded successfully!")
except Exception as e:
    print(f"Error downloading '{filename}': {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'adelaide_property_data.geojson' downloaded successfully!


#Phase 3: Create Visualisation
## Interactive Choropleth Map of Median Price Change

To create an interactive choropleth map that shows price changes over time, we first need to define a start and end date. For this example, we will automatically select the earliest and latest dates available in the `merged_gdf`.

Then, we will:
1.  Filter the `merged_gdf` to get median prices for each suburb at the start and end dates.
2.  Merge these data points to calculate the absolute and percentage price change.
3.  Use the `folium` library to create an interactive map.
4.  Add a choropleth layer colored by the percentage price change.
5.  Include `GeoJsonTooltip` to display suburb details, start price, end price, and price change on mouseover.

### Interactive Date Range Slider

To allow for dynamic selection of the earliest and latest dates for the median price change calculation, we will use `ipywidgets`. This will create a slider interface directly in the notebook output.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np # Import numpy for NaN values
import folium
import pandas as pd
import geopandas as gpd

# Load the separate data files once, globally, for the widget setup
# This ensures that date and suburb lists are consistent with the data
adelaide_suburbs_gdf = gpd.read_file('adelaide_suburbs.geojson')
adelaide_property_df = pd.read_csv('adelaide_property_data.csv', parse_dates=['Date'])


# Get all unique dates from the property DataFrame and sort them
all_dates = pd.to_datetime(adelaide_property_df['Date'].unique())
sorted_dates = sorted(all_dates)

# Create a mapping from index to date and vice versa for the slider
date_to_index = {date: i for i, date in enumerate(sorted_dates)}
index_to_date = {i: date for i, date in enumerate(sorted_dates)}

# Initialize the slider values to the earliest valid date (with min_suburbs_threshold) and the latest date
min_suburbs_threshold = 100
initial_earliest_date = sorted_dates[0]
for d in sorted_dates:
    suburbs_on_date = adelaide_property_df[adelaide_property_df['Date'] == d]['Suburb'].nunique()
    if suburbs_on_date >= min_suburbs_threshold:
        initial_earliest_date = d
        break

initial_earliest_index = date_to_index[initial_earliest_date]
initial_latest_index = date_to_index[sorted_dates[-1]]

# Create a RangeSlider widget
date_slider = widgets.IntRangeSlider(
    value=[initial_earliest_index, initial_latest_index],
    min=1,
    max=len(sorted_dates) - 1,
    step=1,
    description='Date Range:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=False,
    layout=widgets.Layout(width='800px')
)

# Create an output widget to display the map
output_map = widgets.Output()

# Create a Label widget to display the actual date range
date_range_label = widgets.Label(
    value=f"Selected Date Range: {initial_earliest_date.strftime('%Y-%m-%d')} to {sorted_dates[-1].strftime('%Y-%m-%d')}"
)

# Create a Reset button
reset_button = widgets.Button(
    description='Reset Dates & Suburb',
    disabled=False,
    button_style='info',
    tooltip='Reset the date slider and clear suburb selection to initial values',
    icon='undo'
)

# Define unique suburbs for the Combobox
unique_suburbs = sorted(adelaide_property_df['Suburb'].unique().tolist())

# Create the Combobox widget for suburb search
suburb_search = widgets.Combobox(
    placeholder='Start typing or select a suburb',
    options=unique_suburbs,
    description='Search Suburb:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='300px')
)

def create_choropleth_map(comparison_gdf_final, earliest_date, latest_date, center, zoom):
    """
    Generates an interactive Folium choropleth map based on median price changes.

    Args:
        comparison_gdf_final (gpd.GeoDataFrame): GeoDataFrame containing suburb geometries
                                                 and calculated price changes.
        earliest_date (pd.Timestamp): The starting date for the comparison.
        latest_date (pd.Timestamp): The ending date for the comparison.
        center (list): [latitude, longitude] for map center.
        zoom (int): Initial zoom level for the map.

    Returns:
        folium.Map: The generated Folium map object.
    """
    # Create the Folium map centered on Adelaide
    m = folium.Map(location=center, zoom_start=zoom, tiles='CartoDB positron')

    # Add Choropleth layer based on 'Percentage_Change'
    folium.Choropleth(
        geo_data=comparison_gdf_final.to_json(),
        data=comparison_gdf_final,
        columns=['Suburb', 'Percentage_Change'],
        key_on='feature.properties.Suburb',
        fill_color='RdYlGn',
        fill_opacity=0.7,
        line_opacity=0.05,
        legend_name='Median Price Percentage Change (%)',
        highlight=True,
        name='Median Price Change',
        bins=5
    ).add_to(m)

    # Add GeoJson layer with interactive tooltips
    folium.GeoJson(
        comparison_gdf_final.to_json(),
        tooltip=folium.features.GeoJsonTooltip(
            fields=['Suburb', 'Formatted_Start_Sales', 'Formatted_End_Sales', 'Formatted_Start_Median_Price', 'Formatted_End_Median_Price', 'Formatted_Price_Change', 'Formatted_Percentage_Change'],
            aliases=[
                'Suburb:',
                f'Sales ({earliest_date.strftime("%Y-%m-%d")}):',
                f'Sales ({latest_date.strftime("%Y-%m-%d")}):',
                f'Median Price ({earliest_date.strftime("%Y-%m-%d")}):',
                f'Median Price ({latest_date.strftime("%Y-%m-%d")}):',
                'Price Change:',
                'Percentage Change:'
            ],
            localize=True,
            sticky=False,
            labels=True,
            style="""
                background-color: #F0EFEF;
                color: black;
                font-family: sans-serif;
                font-size: 12px;
                padding: 10px;
            """,
            max_width=800
        )
    ).add_to(m)
    return m

def generate_and_display_map(start_date_idx, end_date_idx, selected_suburb_name):
    with output_map:
        clear_output(wait=True)

        earliest_date = index_to_date[start_date_idx]
        latest_date = index_to_date[end_date_idx]

        # Update the date range label
        date_range_label.value = f"Selected Date Range: {earliest_date.strftime('%Y-%m-%d')} to {latest_date.strftime('%Y-%m-%d')}"

        print(f"Creating map for price change between: {earliest_date.strftime('%Y-%m-%d')} and {latest_date.strftime('%Y-%m-%d')}")

        # Filter the property DataFrame for the earliest and latest dates
        start_property_df = adelaide_property_df[adelaide_property_df['Date'] == earliest_date][['Suburb', 'Median Price', 'Sales']].copy()
        start_property_df = start_property_df.rename(columns={'Median Price': 'Start_Median_Price', 'Sales': 'Start_Sales'})

        end_property_df = adelaide_property_df[adelaide_property_df['Date'] == latest_date][['Suburb', 'Median Price', 'Sales']].copy()
        end_property_df = end_property_df.rename(columns={'Median Price': 'End_Median_Price', 'Sales': 'End_Sales'})

        # Merge the start and end property dataframes
        comparison_df = pd.merge(start_property_df, end_property_df, on='Suburb', how='outer') # Use outer to keep all suburbs

        # Merge with the geographic data
        comparison_gdf_final = pd.merge(adelaide_suburbs_gdf, comparison_df, left_on='suburb', right_on='Suburb', how='left')

        # Calculate price change and percentage change for all rows
        comparison_gdf_final['Price_Change'] = comparison_gdf_final['End_Median_Price'] - comparison_gdf_final['Start_Median_Price']
        # Handle division by zero: if Start_Median_Price is 0, Percentage_Change would be inf. Set it to NaN.
        comparison_gdf_final['Percentage_Change'] = (
            (comparison_gdf_final['Price_Change'] / comparison_gdf_final['Start_Median_Price']) * 100
        ).replace([np.inf, -np.inf], np.nan) # Explicitly convert inf to NaN

        # Identify rows where we don't want to color the suburb (due to NaN in calculated Percentage_Change or Median Price)
        no_color_mask = (
            comparison_gdf_final['Percentage_Change'].isna() |
            (comparison_gdf_final['Start_Median_Price'] == 0) | (comparison_gdf_final['Start_Median_Price'].isna()) |
            (comparison_gdf_final['End_Median_Price'] == 0) | (comparison_gdf_final['End_Median_Price'].isna())
        )

        # Set Percentage_Change to NaN for suburbs that should not be colored
        comparison_gdf_final.loc[no_color_mask, 'Percentage_Change'] = np.nan

        # Format columns for tooltip display, handling NaNs and zeros for clarity
        comparison_gdf_final['Formatted_Start_Median_Price'] = comparison_gdf_final['Start_Median_Price'].apply(lambda x: f"${x:,.0f}" if pd.notna(x) and x != 0 else 'N/A')
        comparison_gdf_final['Formatted_End_Median_Price'] = comparison_gdf_final['End_Median_Price'].apply(lambda x: f"${x:,.0f}" if pd.notna(x) and x != 0 else 'N/A')
        comparison_gdf_final['Formatted_Price_Change'] = comparison_gdf_final['Price_Change'].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else 'N/A')
        comparison_gdf_final['Formatted_Percentage_Change'] = comparison_gdf_final['Percentage_Change'].apply(lambda x: f"{x:.2f}%" if pd.notna(x) else 'N/A')
        comparison_gdf_final['Formatted_Start_Sales'] = comparison_gdf_final['Start_Sales'].apply(lambda x: f"{int(x)}" if pd.notna(x) and x != 0 else 'N/A')
        comparison_gdf_final['Formatted_End_Sales'] = comparison_gdf_final['End_Sales'].apply(lambda x: f"{int(x)}" if pd.notna(x) and x != 0 else 'N/A')

        # Ensure the result is a GeoDataFrame
        # comparison_gdf_final = gpd.GeoDataFrame(comparison_df_final, geometry='geometry') # This line might be redundant now, but harmless

        # Determine map center and zoom based on selected suburb
        map_center = [-34.9285, 138.6007] # Default Adelaide center
        map_zoom = 11 # Default zoom

        if selected_suburb_name and selected_suburb_name in unique_suburbs:
            # Use adelaide_suburbs_gdf for reliable suburb geometry lookup
            suburb_geom_entry = adelaide_suburbs_gdf[adelaide_suburbs_gdf['suburb'].str.lower() == selected_suburb_name.lower()]
            if not suburb_geom_entry.empty:
                centroid = suburb_geom_entry.geometry.centroid.iloc[0]
                map_center = [centroid.y, centroid.x] # lat, lon
                map_zoom = 13 # Zoom in a bit for a specific suburb

        # Call the new function to create the map
        m = create_choropleth_map(comparison_gdf_final, earliest_date, latest_date, map_center, map_zoom)
        display(m)

def on_widget_change(change=None):
    # Get current values from both widgets
    current_start_date_idx = date_slider.value[0]
    current_end_date_idx = date_slider.value[1]
    current_selected_suburb = suburb_search.value

    generate_and_display_map(current_start_date_idx, current_end_date_idx, current_selected_suburb)

def reset_slider_dates(b):
    date_slider.value = [initial_earliest_index, initial_latest_index]
    suburb_search.value = '' # Clear suburb search as well on reset
    on_widget_change() # Trigger map update after reset

# Attach observers to both the date slider and the suburb search combobox
date_slider.observe(on_widget_change, names='value')
suburb_search.observe(on_widget_change, names='value')
reset_button.on_click(reset_slider_dates)

# Display the widgets
display(widgets.VBox([date_slider, date_range_label, reset_button, suburb_search]), output_map)

# Generate the initial map when the widget is first displayed
on_widget_change() # Call the combined update function for initial display

Output()